# Lecture 06 · 期末笔试复习

Practice notebook — 待完成

In [ ]:
def prefix_function(pattern):
    """
    pi[i] 表示：
    pattern[:i+1] 的最长相等真前后缀长度
    """
    pi = [0] * len(pattern)

    for i in range(1, len(pattern)):
        j = pi[i - 1]
        while j > 0 and pattern[i] != pattern[j]:
            j = pi[j - 1]
        if pattern[i] == pattern[j]:
            j += 1
        pi[i] = j
    return pi


def kmp_search(text, pattern):
    """
    返回 pattern 在 text 中所有出现位置（0-based）
    """
    if not pattern:
        return []
    pi = prefix_function(pattern)
    j = 0                      # 当前已匹配的模式串长度
    positions = []             # 记录匹配起点
    for i in range(len(text)):
        while j > 0 and text[i] != pattern[j]:
            j = pi[j - 1]
        if text[i] == pattern[j]:
            j += 1
        if j == len(pattern):
            positions.append(i - len(pattern) + 1)
            j = pi[j - 1]      # 继续寻找重叠匹配
    return positions


# 输入
text = input().strip()
pattern = input().strip()

# 查找
ans = kmp_search(text, pattern)

print("匹配位置：", ans)
print("匹配次数：", len(ans))

In [ ]:
import heapq
import sys
from collections import defaultdict

# ------------------------------------------------------------
# 1. Prim 算法（最小生成树）
# ------------------------------------------------------------
def prim_mst(graph, start=0):
    """
    Prim 算法求解最小生成树
    :param graph: 邻接表表示的图，格式 {u: [(v, w), ...], ...}
    :param start: 起始顶点
    :return: (mst_edges, total_weight) 最小生成树的边列表和总权重
    """
    # 记录顶点是否已加入 MST
    visited = set()
    # 优先队列存储 (权重, 当前顶点, 父顶点)
    pq = [(0, start, -1)]
    total_weight = 0
    mst_edges = []

    while pq and len(visited) < len(graph):
        w, u, parent = heapq.heappop(pq)
        if u in visited:
            continue
        # 将顶点 u 加入 MST
        visited.add(u)
        total_weight += w
        if parent != -1:
            mst_edges.append((parent, u, w))
        # 遍历邻居
        for v, weight in graph.get(u, []):
            if v not in visited:
                heapq.heappush(pq, (weight, v, u))
    return mst_edges, total_weight


# ------------------------------------------------------------
# 2. Kruskal 算法（最小生成树）
# ------------------------------------------------------------
class DSU:
    """并查集（Disjoint Set Union）"""
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # 路径压缩
            x = self.parent[x]
        return x

    def union(self, x, y):
        xr, yr = self.find(x), self.find(y)
        if xr == yr:
            return False
        if self.rank[xr] < self.rank[yr]:
            self.parent[xr] = yr
        elif self.rank[xr] > self.rank[yr]:
            self.parent[yr] = xr
        else:
            self.parent[yr] = xr
            self.rank[xr] += 1
        return True


def kruskal_mst(n, edges):
    """
    Kruskal 算法求解最小生成树
    :param n: 顶点个数（顶点编号 0 ~ n-1）
    :param edges: 边列表，每条边为 (u, v, weight)
    :return: (mst_edges, total_weight) 最小生成树的边列表和总权重
    """
    # 按权重升序排序
    edges.sort(key=lambda x: x[2])
    dsu = DSU(n)
    mst_edges = []
    total_weight = 0

    for u, v, w in edges:
        if dsu.union(u, v):
            mst_edges.append((u, v, w))
            total_weight += w
            if len(mst_edges) == n - 1:
                break
    return mst_edges, total_weight


# ------------------------------------------------------------
# 3. Dijkstra 算法（单源最短路径）
# ------------------------------------------------------------
def dijkstra(graph, src):
    """
    Dijkstra 算法求解单源最短路径（非负权图）
    :param graph: 邻接表表示的图，格式 {u: [(v, w), ...], ...}
    :param src: 源点
    :return: dist 列表，dist[v] 表示从 src 到 v 的最短距离，不可达为 float('inf')
    """
    dist = {v: float('inf') for v in graph}
    dist[src] = 0
    # 优先队列存储 (当前距离, 顶点)
    pq = [(0, src)]

    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, w in graph.get(u, []):
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist


# ------------------------------------------------------------
# 4. Floyd 算法（全源最短路径）
# ------------------------------------------------------------
def floyd_warshall(n, edges):
    """
    Floyd-Warshall 算法求解所有顶点对之间的最短路径
    :param n: 顶点个数（顶点编号 0 ~ n-1）
    :param edges: 边列表，每条边为 (u, v, weight)，允许负权但不得有负环
    :return: dist 二维列表，dist[i][j] 为 i 到 j 的最短距离，不可达为 float('inf')
    """
    # 初始化距离矩阵
    dist = [[float('inf')] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
    for u, v, w in edges:
        dist[u][v] = min(dist[u][v], w)  # 若有重边取最小权
        # 若无向图可加上 dist[v][u] = min(dist[v][u], w)

    # 动态规划
    for k in range(n):
        for i in range(n):
            if dist[i][k] == float('inf'):
                continue
            for j in range(n):
                if dist[k][j] == float('inf'):
                    continue
                if dist[i][j] > dist[i][k] + dist[k][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist


# ------------------------------------------------------------
# 示例运行
# ------------------------------------------------------------
if __name__ == "__main__":
    # ---- 构造示例图（无向带权）----
    # 顶点数 5，边列表 (u, v, weight)
    example_edges = [
        (0, 1, 2), (0, 3, 6),
        (1, 2, 3), (1, 3, 8), (1, 4, 5),
        (2, 4, 7),
        (3, 4, 9)
    ]
    n_vertices = 5

    # 构建邻接表（Prim 和 Dijkstra 需要）
    adj_list = defaultdict(list)
    for u, v, w in example_edges:
        adj_list[u].append((v, w))
        adj_list[v].append((u, w))  # 无向图

    print("=== Prim 算法（最小生成树）===")
    mst_edges, total_w = prim_mst(adj_list, start=0)
    print("MST 边:", mst_edges)
    print("总权重:", total_w)

    print("\n=== Kruskal 算法（最小生成树）===")
    mst_edges_k, total_w_k = kruskal_mst(n_vertices, example_edges[:])  # 传递副本
    print("MST 边:", mst_edges_k)
    print("总权重:", total_w_k)

    print("\n=== Dijkstra 算法（单源最短路径，源点=0）===")
    dist = dijkstra(adj_list, 0)
    print("从 0 出发到各点的最短距离:", dict(dist))

    print("\n=== Floyd 算法（全源最短路径）===")
    all_dist = floyd_warshall(n_vertices, example_edges)
    print("任意两点间最短距离矩阵:")
    for row in all_dist:
        print(row)

In [ ]:
import random
import math

# ------------------------------------------------------------
# 1. 直接插入排序 (Insertion Sort)
# ------------------------------------------------------------
def insertion_sort(arr):
    """
    直接插入排序，稳定，时间复杂度 O(n^2)，空间 O(1)
    """
    n = len(arr)
    for i in range(1, n):
        key = arr[i]
        j = i - 1
        # 将比 key 大的元素向后移动
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key


# ------------------------------------------------------------
# 2. 二分插入排序 (Binary Insertion Sort)
# ------------------------------------------------------------
def binary_insertion_sort(arr):
    """
    二分插入排序，稳定，时间复杂度 O(n^2)（比较次数优化但移动次数不变），空间 O(1)
    """
    n = len(arr)
    for i in range(1, n):
        key = arr[i]
        # 二分查找插入位置
        left, right = 0, i - 1
        while left <= right:
            mid = (left + right) // 2
            if arr[mid] > key:
                right = mid - 1
            else:
                left = mid + 1
        # 移动元素 (left 为待插入位置)
        for j in range(i, left, -1):
            arr[j] = arr[j - 1]
        arr[left] = key


# ------------------------------------------------------------
# 3. Shell 排序 (Shell Sort)
# ------------------------------------------------------------
def shell_sort(arr):
    """
    Shell 排序，不稳定，时间复杂度约 O(n^1.3)，空间 O(1)
    使用 Hibbard 增量序列: 2^k - 1，此处简化为 gap //= 2
    """
    n = len(arr)
    gap = n // 2
    while gap > 0:
        # 对每个子序列进行插入排序
        for i in range(gap, n):
            temp = arr[i]
            j = i
            while j >= gap and arr[j - gap] > temp:
                arr[j] = arr[j - gap]
                j -= gap
            arr[j] = temp
        gap //= 2


# ------------------------------------------------------------
# 4. 直接选择排序 (Selection Sort)
# ------------------------------------------------------------
def selection_sort(arr):
    """
    直接选择排序，不稳定，时间复杂度 O(n^2)，空间 O(1)
    """
    n = len(arr)
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            if arr[j] < arr[min_idx]:
                min_idx = j
        if min_idx != i:
            arr[i], arr[min_idx] = arr[min_idx], arr[i]


# ------------------------------------------------------------
# 5. 堆排序 (Heap Sort)
# ------------------------------------------------------------
def heapify(arr, n, i):
    """
    将以 i 为根的子树调整为最大堆
    """
    largest = i
    left = 2 * i + 1
    right = 2 * i + 2
    if left < n and arr[left] > arr[largest]:
        largest = left
    if right < n and arr[right] > arr[largest]:
        largest = right
    if largest != i:
        arr[i], arr[largest] = arr[largest], arr[i]
        heapify(arr, n, largest)


def heap_sort(arr):
    """
    堆排序，不稳定，时间复杂度 O(n log n)，空间 O(1)
    """
    n = len(arr)
    # 建堆（最大堆）
    for i in range(n // 2 - 1, -1, -1):
        heapify(arr, n, i)
    # 依次将堆顶元素与末尾交换，并调整堆
    for i in range(n - 1, 0, -1):
        arr[0], arr[i] = arr[i], arr[0]
        heapify(arr, i, 0)


# ------------------------------------------------------------
# 6. 冒泡排序 (Bubble Sort)
# ------------------------------------------------------------
def bubble_sort(arr):
    """
    冒泡排序，稳定，时间复杂度 O(n^2)，空间 O(1)
    增加 swapped 标志，若某一趟无交换则提前结束
    """
    n = len(arr)
    for i in range(n - 1):
        swapped = False
        for j in range(n - 1 - i):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
        if not swapped:
            break


# ------------------------------------------------------------
# 7. 快速排序 (Quick Sort)
# ------------------------------------------------------------
def partition(arr, low, high):
    """
    分区函数，采用随机基准避免最坏情况
    """
    # 随机选择基准并与最后一个元素交换
    pivot_idx = random.randint(low, high)
    arr[pivot_idx], arr[high] = arr[high], arr[pivot_idx]
    pivot = arr[high]
    i = low - 1
    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    return i + 1


def quick_sort_recursive(arr, low, high):
    if low < high:
        pi = partition(arr, low, high)
        quick_sort_recursive(arr, low, pi - 1)
        quick_sort_recursive(arr, pi + 1, high)


def quick_sort(arr):
    """
    快速排序，不稳定，平均时间复杂度 O(n log n)，最坏 O(n^2)，空间 O(log n)
    使用随机基准，原地排序
    """
    quick_sort_recursive(arr, 0, len(arr) - 1)


# ------------------------------------------------------------
# 8. 归并排序 (Merge Sort)
# ------------------------------------------------------------
def merge(arr, left, mid, right, temp):
    """
    合并两个有序子数组 arr[left..mid] 和 arr[mid+1..right]
    """
    i, j, k = left, mid + 1, left
    while i <= mid and j <= right:
        if arr[i] <= arr[j]:
            temp[k] = arr[i]
            i += 1
        else:
            temp[k] = arr[j]
            j += 1
        k += 1
    while i <= mid:
        temp[k] = arr[i]
        i += 1
        k += 1
    while j <= right:
        temp[k] = arr[j]
        j += 1
        k += 1
    # 将临时数组中的内容复制回原数组
    for idx in range(left, right + 1):
        arr[idx] = temp[idx]

def merge_sort_recursive(arr, left, right, temp):
    if left < right:
        mid = (left + right) // 2
        merge_sort_recursive(arr, left, mid, temp)
        merge_sort_recursive(arr, mid + 1, right, temp)
        merge(arr, left, mid, right, temp)

def merge_sort(arr):
    """
    归并排序，稳定，时间复杂度 O(n log n)，空间 O(n)
    原地排序（借助临时数组）
    """
    temp = [0] * len(arr)
    merge_sort_recursive(arr, 0, len(arr) - 1, temp)


# ------------------------------------------------------------
# 9. 基数排序 (Radix Sort) - LSD，仅适用于非负整数
# ------------------------------------------------------------
def counting_sort_for_radix(arr, exp):
    """
    基数排序的辅助计数排序，按第 exp 位（10^exp）排序
    """
    n = len(arr)
    output = [0] * n
    count = [0] * 10  # 十进制数字 0~9

    # 统计当前位的出现次数
    for i in range(n):
        digit = (arr[i] // exp) % 10
        count[digit] += 1

    # 转换为累积计数（稳定排序的关键）
    for i in range(1, 10):
        count[i] += count[i - 1]

    # 从后向前构建输出数组
    for i in range(n - 1, -1, -1):
        digit = (arr[i] // exp) % 10
        output[count[digit] - 1] = arr[i]
        count[digit] -= 1

    # 复制回原数组
    for i in range(n):
        arr[i] = output[i]


def radix_sort(arr):
    """
    基数排序 (LSD)，稳定，时间复杂度 O(d*(n+r))，其中 d 为最大位数，r=10，空间 O(n+r)
    适用于非负整数数组，负数需要额外处理（此处不支持）
    """
    if not arr:
        return
    max_val = max(arr)
    exp = 1
    while max_val // exp > 0:
        counting_sort_for_radix(arr, exp)
        exp *= 10


# ------------------------------------------------------------
# 测试与验证
# ------------------------------------------------------------
def test_sort(sort_func, arr):
    """
    测试单个排序函数，打印排序前后的部分信息
    """
    original = arr[:]
    sort_func(arr)
    print(f"{sort_func.__name__:25} | 排序{'成功' if arr == sorted(original) else '失败'}")
    # 可选：显示前10个元素
    # print(f"  示例: {original[:10]} -> {arr[:10]}")


if __name__ == "__main__":
    # 生成随机测试数组
    test_arr = [random.randint(0, 1000) for _ in range(100)]
    # 所有排序算法列表（按顺序）
    sorts = [
        insertion_sort,
        binary_insertion_sort,
        shell_sort,
        selection_sort,
        heap_sort,
        bubble_sort,
        quick_sort,
        merge_sort,
        radix_sort
    ]

    print("=" * 60)
    print("排序算法测试 (数组长度 = {})".format(len(test_arr)))
    print("=" * 60)

    for sort in sorts:
        # 每次使用原数组的副本，避免影响其他排序
        arr_copy = test_arr[:]
        test_sort(sort, arr_copy)

    # 额外验证所有排序结果是否一致
    baseline = sorted(test_arr)
    print("\n基准排序结果 (前20个):", baseline[:20])